# SahayakLLM — Notebook 02 (Kaggle): QLoRA Fine-Tuning, C2 Evaluation, Cost, Adapter Release (+ optional Data Budget)

**Tasks:** T21 (QLoRA config) · T22 (train, Figure 2) · T23 (C2 vs B0/B1) · T24 (inference cost) · T26 (adapter + model card + upload) · T25 (data-budget runs, optional section at the end)
**Expected time on a Kaggle T4:** main training ≈ 2.5–3 h, evaluation ≈ 1 h; data budget ≈ 4 h more (off by default)

## One-time Kaggle setup
1. **Upload the Notebook 01 outputs as a Kaggle Dataset.** From your Drive folder `SahayakLLM/`, take these sub-folders (leave out `hf_cache/` and `preds/`): `config/`, `data/splits/`, `prompts/`, `eval/`, `src/`, `results/`. Zip them (keeping the folder names) → *Kaggle → Datasets → New Dataset → upload*. Any dataset name works; the notebook finds it automatically.
2. **Create a new Notebook** (File → *Import notebook* → this file) and in the right-hand panel set:
   * **Accelerator:** *GPU T4 ×2* (the notebook uses one GPU; avoid *P100*, whose 4-bit support is unreliable)
   * **Internet:** *On* (needed to download the model from Hugging Face; requires a phone-verified account)
   * **Add Data:** attach your dataset from step 1
   * **Secrets (optional, for the Hub upload):** *Add-ons → Secrets* → add `HF_TOKEN` (write access) and tick it for this notebook
3. Click **Run All**. Everything is written to `/kaggle/working/SahayakLLM/` (visible in the *Output* tab); the Hugging Face model cache goes to temporary disk so it does not count toward the 20 GB output limit.

## Sessions and persistence
* Files in `/kaggle/working` survive only while the session is alive and in a **saved version**. For long runs use **Save Version → Save & Run All (Commit)**, so the outputs (adapter, predictions, results log) are kept.
* If a session ends early, open the next one, **attach the previous version's output as an additional input** (*Add Data → Notebook Output*) next to your dataset, and Run All again: the notebook merges all inputs, resumes training from the last checkpoint and skips every finished step.
* Run the optional data-budget section (set `RUN_DATA_BUDGET = True`) as a **separate session** after the main run.

**This notebook loads the saved data and verifies the manifest hash; it never recomputes the data.**

**Hugging Face upload (T26, optional):** set `HF_ADAPTER_REPO` below, e.g. `yourname/sahayakllm-qlora`. If it is empty or no `HF_TOKEN` secret exists, the upload step is skipped and the adapter stays in `/kaggle/working` (a zip is also created for download).

## 0. Configuration

In [ ]:
# ===================== EDIT THESE =====================
WORK_ROOT = "/kaggle/working/SahayakLLM"             # outputs (Output tab)
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"            # must match Notebook 01
SEED = 42

# QLoRA (brief, Section 11)
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LEARNING_RATE, WARMUP_RATIO = 2e-4, 0.03
PER_DEVICE_BS, GRAD_ACCUM = 4, 4                      # effective batch 16
EPOCHS_MAIN = 1                                      # full data
EVAL_EVERY = 50                                      # train/val loss logged every 50 steps
VAL_RECORDS = 320                                    # validation records used during training
SAVE_EVERY = 500                                     # checkpoint (session may end)

# Evaluation
TEST_LIMIT = None            # None = full test split (use same value as Notebook 01)
RUN_HELDOUT = True           # C2 on the held-out-intent set
PPL_N = 300
LATENCY_N = 20
GEN_BS_ZERO_SHOT, GEN_BS_ROUTE = 32, 48

# Release (T26)
HF_ADAPTER_REPO = ""         # e.g. "yourname/sahayakllm-qlora" ; "" = skip upload
VERIFY_HF_LOAD = True        # reload the uploaded adapter from the Hub to prove it works on a fresh runtime

# Data budget (T25) - leave False for the first run; set True afterwards
RUN_DATA_BUDGET = False
BUDGET_PLAN = {1000: 3, 2000: 3, 5000: 2, 10000: 1}  # training rows -> epochs (brief: 2-3 epochs for small budgets)
# ======================================================

## 1. Setup (Kaggle)

In [ ]:
import os, sys, json, re, math, time, hashlib, random, shutil, datetime, gc, inspect, warnings
os.environ["CUDA_VISIBLE_DEVICES"] = "0"      # T4 x2: use ONE GPU (Trainer would otherwise wrap the 4-bit model in DataParallel)
from pathlib import Path
assert Path("/kaggle/input").exists(), "This notebook is written for Kaggle."

WORK = Path(WORK_ROOT); WORK.mkdir(parents=True, exist_ok=True)
ROOT = WORK
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Hugging Face cache on temporary disk (not part of the 20 GB output limit)
HF_CACHE = Path("/kaggle/temp/hf_cache") if Path("/kaggle/temp").exists() else Path("/tmp/hf_cache")
HF_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_CACHE)          # must be set BEFORE transformers is imported

# Optional token (faster, authenticated downloads + needed for the upload step)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ.setdefault("HF_TOKEN", UserSecretsClient().get_secret("HF_TOKEN"))
except Exception:
    pass

# Locate every attached input that contains the project folders (your dataset and/or a previous version's output)
found = []
for d in range(1, 7):                              # dataset folders can be nested a few levels deep
    prefix = "*/" * d
    found += [m.parent.parent.parent for m in Path("/kaggle/input").glob(prefix + "data/splits/manifest.json")]
    found += [m.parent for m in Path("/kaggle/input").glob(prefix + "adapters")]
roots = sorted(set(found))
assert roots, ("No project data found in /kaggle/input. Add the dataset you made from the Notebook 01 outputs "
               "(config/, data/splits/, prompts/, eval/, src/, results/) via 'Add Data'.")

def merge(src, dst):
    n = 0
    for p in src.rglob("*"):
        if p.is_dir() or "hf_cache" in p.parts or "__pycache__" in p.parts:
            continue
        t = dst / p.relative_to(src)
        if not t.exists() or p.stat().st_mtime > t.stat().st_mtime:
            t.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(p, t); n += 1
    return n
for r in roots:
    print(f"merged {merge(r, WORK):4d} files from {r}")

assert (ROOT / "data/splits/manifest.json").exists(), "data/splits/manifest.json missing from the uploaded dataset"
for need in ["eval/metrics.py", "eval/generation.py", "src/data/split.py", "eval/results_template.csv",
             "prompts/system.txt", "config/data_config.json"]:
    assert (ROOT / need).exists(), f"missing {need} in the uploaded dataset"
D = {k: ROOT / k for k in ["config", "data/splits", "prompts", "preds", "results", "figures", "reports", "docs", "adapters"]}
for p in D.values():
    p.mkdir(parents=True, exist_ok=True)

# Internet check (model + metric models come from the Hugging Face Hub)
import urllib.request
try:
    urllib.request.urlopen("https://huggingface.co", timeout=10)
    print("internet OK")
except Exception as e:
    raise RuntimeError("No internet. Turn Internet ON in the notebook settings (right panel).") from e

In [ ]:
# Kaggle already ships torch/transformers; install only what may be missing or too old.
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "peft", "bitsandbytes", "accelerate",
                "rouge_score", "bert_score", "evaluate"], check=False)

In [ ]:
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore", category=FutureWarning)
assert torch.cuda.is_available(), "No GPU: Settings (right panel) > Accelerator > GPU T4 x2"
GPU_NAME = torch.cuda.get_device_name(0)
CC = torch.cuda.get_device_capability(0)
print("GPU:", GPU_NAME, "| compute capability", CC, "| visible GPUs:", torch.cuda.device_count())
if CC[0] < 7:
    print("WARNING: this GPU is older than a T4 (e.g. P100); 4-bit bitsandbytes may fail. Choose GPU T4 x2.")
import transformers, peft, bitsandbytes
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__, "| bitsandbytes", bitsandbytes.__version__)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

sys.path[:0] = [str(ROOT / "src/data"), str(ROOT / "eval")]
from split import verify_manifest, read_jsonl
from metrics import (intent_metrics, parse_route_json, rouge_l, bertscore_f1, response_perplexity,
                     placeholder_leakage, latency_stats, has_placeholder)
from generation import (load_model_4bit, build_response_messages, build_route_messages, generate_chat,
                        latency_benchmark)

def free_gpu():
    gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ---- load the saved data + frozen prompt; verify the manifest hash (never recompute the data) ----
SPLIT_DIR = D["data/splits"]
manifest = verify_manifest(str(SPLIT_DIR))
MANIFEST_HASH = manifest["manifest_hash"]
tr, va, te, ho, mini = [read_jsonl(SPLIT_DIR / f"{n}.jsonl") for n in ("train", "val", "test", "held_out_test", "eval_mini")]
sft_train = [json.loads(l) for l in open(SPLIT_DIR / "sft_train.jsonl", encoding="utf-8")]
sft_val = [json.loads(l) for l in open(SPLIT_DIR / "sft_val.jsonl", encoding="utf-8")]

SYSTEM_PROMPT = (D["prompts"] / "system.txt").read_text(encoding="utf-8")
ROUTER_SYSTEM = (D["prompts"] / "router_system.txt").read_text(encoding="utf-8")
frozen = json.loads((D["prompts"] / "frozen.json").read_text())
sft_man = json.loads((SPLIT_DIR / "sft_manifest.json").read_text())
PROMPT_HASH = hashlib.sha256(SYSTEM_PROMPT.encode()).hexdigest()
assert PROMPT_HASH == frozen["system_sha256"] == sft_man["system_prompt_sha256"], "frozen prompt changed since Notebook 01"
assert sft_man["data_manifest_hash"] == MANIFEST_HASH, "SFT files do not match the split manifest"

cfg = json.loads((D["config"] / "data_config.json").read_text())
MAX_SEQ, MAX_NEW_RESP, MAX_NEW_ROUTE = cfg["max_seq_length"], cfg["max_new_tokens_resp"], cfg["max_new_tokens_route"]
ALL_INTENTS = sorted(set(tr.intent) | set(va.intent) | set(te.intent) | set(ho.intent))

RESULT_COLS = list(pd.read_csv(ROOT / "eval/results_template.csv").columns)
RESULTS_PATH = D["results"] / "results_log.csv"
def log_row(row):
    df = pd.read_csv(RESULTS_PATH) if RESULTS_PATH.exists() else pd.DataFrame(columns=RESULT_COLS)
    df = df[df["run_id"] != row["run_id"]]
    row = {c: row.get(c, np.nan) for c in RESULT_COLS}
    row["date"] = str(datetime.datetime.now())[:19]; row["seed"] = SEED
    pd.concat([df, pd.DataFrame([row])], ignore_index=True).to_csv(RESULTS_PATH, index=False)

print(f"manifest verified {MANIFEST_HASH[:16]} | prompt {frozen['chosen_variant']} ({PROMPT_HASH[:12]}) | max_seq_length {MAX_SEQ}")
print(f"sft_train {len(sft_train)} records ({sft_man['routing_share']:.1%} routing) | sft_val {len(sft_val)} | train/val/test/held-out rows: {len(tr)}/{len(va)}/{len(te)}/{len(ho)}")

## 2. Training machinery (T21)
* **QLoRA:** 4-bit NF4 + double quantisation, float16 compute (T4 has no bfloat16); LoRA r=16, α=32, dropout 0.05 on all seven linear projections; lr 2e-4 cosine, 3 % warm-up; per-device batch 4 × accumulation 4 = effective 16; gradient checkpointing on.
* **Loss on assistant tokens only:** every record is rendered with the tokenizer's own chat template; all prompt tokens (system + user) get label −100, and the loss covers the assistant reply plus the closing `<|im_end|>`. This is done explicitly (not via a TRL option), so it works with any library version.
* One adapter serves both tasks (response + routing records in the same run).

In [ ]:
from transformers import Trainer, TrainingArguments
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

END = "<|im_end|>"

def load_tok():
    from transformers import AutoTokenizer
    t = AutoTokenizer.from_pretrained(BASE_MODEL)
    if t.pad_token is None:
        t.pad_token = t.eos_token
    return t
tok = load_tok()
assert tok.eos_token == END, f"unexpected eos token {tok.eos_token}"

def encode(rec):
    msgs = rec["messages"]
    ptxt = tok.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True)
    p = tok(ptxt, add_special_tokens=False)["input_ids"]
    r = tok(msgs[-1]["content"] + END, add_special_tokens=False)["input_ids"]
    return {"input_ids": (p + r)[:MAX_SEQ], "labels": ([-100] * len(p) + r)[:MAX_SEQ], "truncated": len(p) + len(r) > MAX_SEQ,
            "n_loss": max(0, min(len(r), MAX_SEQ - len(p)))}

class ListDS(torch.utils.data.Dataset):
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return {"input_ids": self.items[i]["input_ids"], "labels": self.items[i]["labels"]}

def collate(batch):
    L = max(len(b["input_ids"]) for b in batch)
    pad = tok.pad_token_id
    return {"input_ids": torch.tensor([b["input_ids"] + [pad] * (L - len(b["input_ids"])) for b in batch]),
            "attention_mask": torch.tensor([[1] * len(b["input_ids"]) + [0] * (L - len(b["input_ids"])) for b in batch]),
            "labels": torch.tensor([b["labels"] + [-100] * (L - len(b["labels"])) for b in batch])}

def make_ds(records, name):
    enc = [encode(r) for r in records]
    trunc = np.mean([e["truncated"] for e in enc]); empty = sum(e["n_loss"] == 0 for e in enc)
    print(f"{name}: {len(enc)} records | truncated {trunc:.2%} | records with no loss tokens: {empty}")
    assert empty == 0
    return ListDS(enc)

LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

def count_params(model):
    try:
        return model.get_nb_trainable_parameters()
    except Exception:
        tr_ = sum(p.numel() for p in model.parameters() if p.requires_grad)
        return tr_, sum(p.numel() for p in model.parameters())

def train_adapter(name, records, epochs, val_records):
    out = D["adapters"] / name
    final = out / "final"
    if (final / "adapter_model.safetensors").exists():
        print(f"[{name}] already trained -> skipping ({final})"); return json.loads((out / "training_summary.json").read_text())
    ckpt_dir = out / "checkpoints"; ckpt_dir.mkdir(parents=True, exist_ok=True)
    train_ds, val_ds = make_ds(records, f"{name} train"), make_ds(val_records[:VAL_RECORDS], f"{name} val")

    _, model = load_model_4bit(BASE_MODEL)
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    lora = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, task_type="CAUSAL_LM", target_modules=LORA_TARGETS)
    model = get_peft_model(model, lora)
    model.config.use_cache = False
    model.print_trainable_parameters()
    n_tr, n_all = count_params(model)

    steps_total = math.ceil(len(train_ds) / (PER_DEVICE_BS * GRAD_ACCUM)) * epochs
    warmup_steps = max(1, int(WARMUP_RATIO * steps_total))     # works in old and new transformers (warmup_ratio was removed in v5)
    kw = dict(output_dir=str(ckpt_dir), per_device_train_batch_size=PER_DEVICE_BS, per_device_eval_batch_size=8,
              gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LEARNING_RATE, lr_scheduler_type="cosine",
              warmup_steps=warmup_steps, num_train_epochs=epochs, fp16=True, logging_steps=EVAL_EVERY, eval_steps=EVAL_EVERY,
              save_strategy="steps", save_steps=SAVE_EVERY, save_total_limit=2, optim="paged_adamw_8bit", max_grad_norm=0.3,
              gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False}, report_to="none",
              seed=SEED, data_seed=SEED, remove_unused_columns=False, dataloader_num_workers=0)
    kw["eval_strategy" if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters else "evaluation_strategy"] = "steps"
    ok = inspect.signature(TrainingArguments.__init__).parameters
    dropped = [k for k in kw if k not in ok]
    if dropped:
        print("NOTE: this transformers version does not accept", dropped, "- ignoring them")
    kw = {k: v for k, v in kw.items() if k in ok}
    trainer = Trainer(model=model, args=TrainingArguments(**kw), train_dataset=train_ds, eval_dataset=val_ds, data_collator=collate)

    ckpts = sorted(ckpt_dir.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[1]))
    resume = str(ckpts[-1]) if ckpts else None
    print(f"[{name}] steps/epoch ~{math.ceil(len(train_ds) / (PER_DEVICE_BS * GRAD_ACCUM))} x {epochs} epoch(s) | resume from: {resume}")
    t0 = time.time()
    result = trainer.train(resume_from_checkpoint=resume)
    secs = time.time() - t0

    logs = pd.DataFrame(trainer.state.log_history)
    logs.to_csv(out / "loss_log.csv", index=False)
    model.save_pretrained(final); tok.save_pretrained(final)
    ev = logs[logs.get("eval_loss").notna()] if "eval_loss" in logs else pd.DataFrame()
    summary = {"name": name, "train_rows": len(records), "epochs": epochs, "steps": int(trainer.state.global_step),
               "train_seconds_this_session": round(secs, 1), "final_train_loss": float(result.training_loss),
               "best_eval_loss": float(ev["eval_loss"].min()) if len(ev) else None,
               "last_eval_loss": float(ev["eval_loss"].iloc[-1]) if len(ev) else None,
               "trainable_params": int(n_tr), "total_params": int(n_all), "trainable_pct": 100 * n_tr / n_all,
               "max_seq_length": MAX_SEQ, "data_manifest_hash": MANIFEST_HASH, "prompt_sha256": PROMPT_HASH}
    (out / "training_summary.json").write_text(json.dumps(summary, indent=2))
    del trainer, model; free_gpu()
    return summary

## 3. Train the main adapter on the full training split (T21–T22)

In [ ]:
main = train_adapter("C2", sft_train, EPOCHS_MAIN, sft_val)
(D["config"] / "qlora_config.json").write_text(json.dumps(
    {"base_model": BASE_MODEL, "quantization": "nf4 + double quant, fp16 compute", "lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
     "lora_dropout": LORA_DROPOUT, "target_modules": LORA_TARGETS, "learning_rate": LEARNING_RATE, "schedule": "cosine",
     "warmup_ratio": WARMUP_RATIO, "per_device_batch": PER_DEVICE_BS, "grad_accum": GRAD_ACCUM,
     "effective_batch": PER_DEVICE_BS * GRAD_ACCUM, "epochs": EPOCHS_MAIN, "max_seq_length": MAX_SEQ,
     "optimizer": "paged_adamw_8bit", "max_grad_norm": 0.3, "loss": "assistant tokens only (+ <|im_end|>)",
     "trainable_params": main["trainable_params"], "total_params": main["total_params"],
     "trainable_pct": main["trainable_pct"], "seed": SEED}, indent=2))
print(json.dumps(main, indent=2))
print(f"\nTRAINABLE PARAMETERS: {main['trainable_params']:,} of {main['total_params']:,} ({main['trainable_pct']:.2f}%)  <- record in the report")

In [ ]:
# Figure 2 - loss curves
logs = pd.read_csv(D["adapters"] / "C2" / "loss_log.csv")
fig, ax = plt.subplots(figsize=(8, 4.2))
t = logs[logs["loss"].notna()] if "loss" in logs else pd.DataFrame()
e = logs[logs["eval_loss"].notna()] if "eval_loss" in logs else pd.DataFrame()
ax.plot(t["step"], t["loss"], label="train loss", color="#3b6ea5")
ax.plot(e["step"], e["eval_loss"], label="validation loss", color="#c0392b", marker="o", ms=3)
ax.set_xlabel("optimizer step"); ax.set_ylabel("loss (assistant tokens)"); ax.set_title("Figure 2 - QLoRA loss curves (C2)")
ax.legend(); ax.grid(alpha=.3); plt.tight_layout()
plt.savefig(D["figures"] / "fig2_loss_curve.png", dpi=170); plt.show()
print(f"best validation loss {e['eval_loss'].min():.4f} at step {int(e.loc[e['eval_loss'].idxmin(), 'step'])} | last {e['eval_loss'].iloc[-1]:.4f}")

## 4. Inference cost: base vs fine-tuned (T24)
Same GPU, same 20 test prompts, batch size 1, greedy decoding, same generation cap. Writes `results/systems_metrics.csv`. The C2 model is left loaded for the evaluation below.

In [ ]:
FINAL = D["adapters"] / "C2" / "final"
lat_prompts = [build_response_messages(SYSTEM_PROMPT, i) for i in te.sort_values("row_id").instruction.sample(LATENCY_N + 1, random_state=SEED)]

def measure(label, adapter):
    _, m = load_model_4bit(BASE_MODEL, adapter_path=adapter)
    lat = latency_benchmark(m, tok, lat_prompts, MAX_NEW_RESP, n=LATENCY_N)
    s = latency_stats(lat["seconds"])
    row = {"model": label, "gpu": GPU_NAME, "n_probes": LATENCY_N, "p50_s": s["p50_s"], "p95_s": s["p95_s"],
           "tokens_per_s": lat["tokens_per_s"], "peak_gpu_gb": lat["peak_gpu_gb"],
           "adapter_size_mb": (sum(f.stat().st_size for f in Path(adapter).glob("*.safetensors")) / 1e6) if adapter else 0.0}
    return row, m

row_base, m = measure("base (B0 weights)", None); del m; free_gpu()
row_c2, model = measure("fine-tuned (C2 adapter)", str(FINAL))
sysm = pd.DataFrame([row_base, row_c2]); sysm.to_csv(D["results"] / "systems_metrics.csv", index=False)
display(sysm.round(3))

## 5. Evaluate C2 on eval-mini, test and held-out intents (T23)
Same harness, prompt, generation cap and greedy decoding as B0/B1 in Notebook 01. Predictions are cached in the output folder (resumable).

In [ ]:
def run_config(config, split_name, frame, limit=None, bs_resp=GEN_BS_ZERO_SHOT):
    sub = frame if limit is None else frame.sample(min(limit, len(frame)), random_state=SEED)
    sub = sub.sort_values("row_id").reset_index(drop=True)
    ids = sub["row_id"].tolist()
    base = f"{config}_{split_name}"
    resp_msgs = [build_response_messages(SYSTEM_PROMPT, ins) for ins in sub.instruction]
    route_msgs = [build_route_messages(ROUTER_SYSTEM, ins) for ins in sub.instruction]
    resp = generate_chat(model, tok, resp_msgs, ids, MAX_NEW_RESP, bs_resp, str(D["preds"] / f"{base}_response.jsonl"))
    route = generate_chat(model, tok, route_msgs, ids, MAX_NEW_ROUTE, GEN_BS_ROUTE, str(D["preds"] / f"{base}_route.jsonl"))
    outs = [r["output"] for r in resp]
    pred_intent = [parse_route_json(r["output"], ALL_INTENTS)[0] for r in route]
    im = intent_metrics(sub.intent, pred_intent, labels=sorted(set(sub.intent)))
    rl, rl_each = rouge_l(outs, sub.response.tolist())
    bf, bf_each = bertscore_f1(outs, sub.response.tolist())
    pd.DataFrame({"row_id": ids, "intent_gold": sub.intent, "intent_pred": pred_intent, "rougeL": rl_each,
                  "bertscore_f1": bf_each, "has_placeholder": [has_placeholder(o) for o in outs]}
                 ).to_csv(D["preds"] / f"{base}_scored.csv", index=False)
    return {"run_id": base, "config": config, "split": split_name, "data_manifest_hash": MANIFEST_HASH,
            "prompt_hash": PROMPT_HASH[:12], "n": len(sub), "intent_acc": im["intent_acc"],
            "intent_macro_f1": im["intent_macro_f1"], "invalid_json_rate": im["invalid_rate"], "rougeL": rl,
            "bertscore_f1": bf, "leak_pct": placeholder_leakage(outs),
            "avg_resp_tokens": float(np.mean([r["n_new_tokens"] for r in resp])),
            "trunc_pct": 100 * float(np.mean([r["n_new_tokens"] >= MAX_NEW_RESP for r in resp]))}, sub

def fmt(r): return {k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()}

In [ ]:
r, _ = run_config("C2", "evalmini", mini); log_row(r); print(fmt(r))

In [ ]:
# long step (~30-40 min); resumable
r, sub_test = run_config("C2", "test", te, limit=TEST_LIMIT)
ppl_items = [(build_response_messages(SYSTEM_PROMPT, i), x) for i, x in zip(sub_test.instruction[:PPL_N], sub_test.response[:PPL_N])]
r["perplexity"] = response_perplexity(model, tok, ppl_items, batch_size=4)
r.update({"p50_s": row_c2["p50_s"], "p95_s": row_c2["p95_s"], "tokens_per_s": row_c2["tokens_per_s"], "peak_gpu_gb": row_c2["peak_gpu_gb"]})
log_row(r); print(fmt(r))

In [ ]:
if RUN_HELDOUT:
    r, _ = run_config("C2", "heldout", ho, limit=TEST_LIMIT)
    log_row(r); print(fmt(r))
else:
    print("Skipped (RUN_HELDOUT=False)")

In [ ]:
# E1 table (partial): B0 / B1 / C2
res = pd.read_csv(RESULTS_PATH)
cols = ["run_id", "n", "intent_acc", "intent_macro_f1", "invalid_json_rate", "rougeL", "bertscore_f1", "perplexity",
        "leak_pct", "avg_resp_tokens", "trunc_pct", "p50_s", "p95_s", "tokens_per_s", "peak_gpu_gb"]
e1 = res[res.run_id.str.match(r"^(B0|B1|C2)_(evalmini|test|heldout)$")][cols].sort_values("run_id")
e1.to_csv(D["results"] / "e1_partial_table.csv", index=False); display(e1.round(4))

t = res[res.split == "test"].set_index("config").loc[[c for c in ["B0", "B1", "C2"] if c in set(res[res.split == "test"].config)]]
fig, ax = plt.subplots(1, 4, figsize=(16, 3.8))
for a, col, title in zip(ax, ["intent_macro_f1", "bertscore_f1", "rougeL", "leak_pct"],
                         ["Intent macro-F1", "BERTScore-F1", "ROUGE-L", "Placeholder leakage (%)"]):
    t[col].plot.bar(ax=a, color=["#9aa5b1", "#6c8ebf", "#2e7d32"][: len(t)], rot=0, title=title + " (test)")
    for p in a.patches:
        a.annotate(f"{p.get_height():.3f}", (p.get_x() + p.get_width() / 2, p.get_height()), ha="center", va="bottom")
plt.tight_layout(); plt.savefig(D["figures"] / "fig_c2_vs_baselines_test.png", dpi=150); plt.show()

## 6. Adapter release: model card, Hub upload, load check (T26)
Writes `MODEL_CARD.md` (also saved as the adapter's `README.md` and to `docs/`). Upload is skipped when `HF_ADAPTER_REPO` is empty.

In [ ]:
res = pd.read_csv(RESULTS_PATH).set_index("run_id")
def g(run, col, nd=3):
    try:
        v = res.loc[run, col]; return "n/a" if pd.isna(v) else f"{v:.{nd}f}"
    except KeyError:
        return "n/a"
card = f"""---
base_model: {BASE_MODEL}
library_name: peft
license: apache-2.0
tags: [qlora, lora, customer-support, e-commerce, peft]
datasets: [bitext/Bitext-customer-support-llm-chatbot-training-dataset]
---
# SahayakLLM - QLoRA adapter for e-commerce customer support

One LoRA adapter on **{BASE_MODEL}** (4-bit NF4) trained for two tasks: (a) customer-support **response generation** and
(b) **routing** to JSON `{{"intent": ..., "category": ...}}` (about 20% of records, marked by a separate system prompt).

## Training data
Bitext Customer Support LLM Chatbot Training Dataset (Bitext Innovations, CDLA-Sharing-1.0), cleaned and split with a fixed seed
({SEED}); stratified 80/10/10 over the training intents, plus held-out intents never used for training
({', '.join(manifest['held_out_intents'])}). Data manifest hash: `{MANIFEST_HASH}`.
Training records: {main['train_rows']:,} (response + routing). KharidiKart is a fictional company.

## Hyperparameters
LoRA r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}, all linear projections; lr {LEARNING_RATE} cosine, {WARMUP_RATIO:.0%} warm-up;
batch {PER_DEVICE_BS}x{GRAD_ACCUM}={PER_DEVICE_BS * GRAD_ACCUM}; {EPOCHS_MAIN} epoch(s); max sequence length {MAX_SEQ}; fp16 compute; paged AdamW 8-bit;
loss on assistant tokens only. Trainable parameters: {main['trainable_params']:,} ({main['trainable_pct']:.2f}%).

## Results (test split, greedy decoding)
| Config | Intent macro-F1 | BERTScore-F1 | ROUGE-L | Placeholder leakage % |
|---|---|---|---|---|
| B0 zero-shot | {g('B0_test','intent_macro_f1')} | {g('B0_test','bertscore_f1')} | {g('B0_test','rougeL')} | {g('B0_test','leak_pct',1)} |
| B1 few-shot | {g('B1_test','intent_macro_f1')} | {g('B1_test','bertscore_f1')} | {g('B1_test','rougeL')} | {g('B1_test','leak_pct',1)} |
| **C2 (this adapter)** | {g('C2_test','intent_macro_f1')} | {g('C2_test','bertscore_f1')} | {g('C2_test','rougeL')} | {g('C2_test','leak_pct',1)} |

Held-out intents (C2): intent macro-F1 {g('C2_heldout','intent_macro_f1')}, BERTScore-F1 {g('C2_heldout','bertscore_f1')}.

## Usage
```python
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
base = AutoModelForCausalLM.from_pretrained("{BASE_MODEL}", quantization_config=bnb, device_map={{"": 0}})
model = PeftModel.from_pretrained(base, "<this-repo>")
```
Use the exact system prompts of the project (`prompts/system.txt` for responses, `prompts/router_system.txt` for routing).

## Limitations
* Trained on synthetic, templated support dialogues for a **fictional** retailer; not validated on real customers.
* Responses keep `{{{{placeholder}}}}` slots instead of real facts; facts must come from tools and be checked by a verifier.
* Generic policies only: the model can state invented policy numbers if it is not given grounded evidence.
* The routing label list is given in the prompt; accuracy on unseen intents is reported separately (held-out set).
* English only; 1.5B-parameter model with 4-bit quantisation; not for production or medical/legal/financial advice.
"""
(FINAL / "README.md").write_text(card, encoding="utf-8")
(D["docs"] / "MODEL_CARD.md").write_text(card, encoding="utf-8")
print(card[:1600], "...")

In [ ]:
UPLOADED = False
token = os.environ.get("HF_TOKEN")
if HF_ADAPTER_REPO.strip() and token:
    from huggingface_hub import HfApi
    api = HfApi(token=token)
    api.create_repo(HF_ADAPTER_REPO, private=True, exist_ok=True)
    api.upload_folder(folder_path=str(FINAL), repo_id=HF_ADAPTER_REPO, commit_message="SahayakLLM QLoRA adapter (C2)")
    UPLOADED = True
    print("uploaded (private):", f"https://huggingface.co/{HF_ADAPTER_REPO}")
elif HF_ADAPTER_REPO.strip():
    print("HF_ADAPTER_REPO is set but no HF_TOKEN secret was found (Add-ons > Secrets) -> upload skipped.")
else:
    print("HF_ADAPTER_REPO is empty -> upload skipped.")
print("Adapter folder:", FINAL)
# zip for download from the Output tab
shutil.make_archive(str(WORK.parent / "C2_adapter"), "zip", str(FINAL))
print("zip:", WORK.parent / "C2_adapter.zip")

In [ ]:
# Prove the Hub copy loads on a fresh runtime path: reload base + adapter from the Hub and answer one prompt.
if UPLOADED and VERIFY_HF_LOAD:
    del model; free_gpu()
    _, hub_model = load_model_4bit(BASE_MODEL, adapter_path=HF_ADAPTER_REPO)
    q = build_response_messages(SYSTEM_PROMPT, "I want to cancel my order, how do I do that?")
    print(generate_chat(hub_model, tok, [q], ["hub_check"], 120, 1, "/dev/null")[0]["output"])
    del hub_model; free_gpu()

## 7. (Optional) Data-budget runs — T25
Set `RUN_DATA_BUDGET = True` in the config and run from the top (or only this section after Section 1–2 cells). Trains adapters on **1k, 2k, 5k, 10k** training rows (stratified by intent, each row's response record plus its routing record when it has one, so the routing share stays ≈ 20 %), skipping any that already exist; evaluates each on **eval-mini**; plots a partial **Figure 3** with the full-data C2 and B0/B1 as references. Epochs per budget are set in `BUDGET_PLAN` (≈ 4 h total).

In [ ]:
if RUN_DATA_BUDGET:
    from sklearn.model_selection import train_test_split
    for n_rows, ep in BUDGET_PLAN.items():
        sub_rows, _ = train_test_split(tr, train_size=n_rows, stratify=tr["intent"], random_state=SEED)
        ids = set(sub_rows.row_id)
        recs = [r for r in sft_train if r["id"] in ids]
        print(f"\n=== budget {n_rows} rows -> {len(recs)} records, {ep} epoch(s) ===")
        globals().pop("model", None); free_gpu()
        train_adapter(f"C2_n{n_rows}", recs, ep, sft_val)
        if (D["preds"] / f"C2_n{n_rows}_evalmini_scored.csv").exists():
            continue
        _, model = load_model_4bit(BASE_MODEL, adapter_path=str(D["adapters"] / f"C2_n{n_rows}" / "final"))
        r, _ = run_config(f"C2_n{n_rows}", "evalmini", mini); log_row(r); print(fmt(r))
        del model; free_gpu()
else:
    print("Skipped (RUN_DATA_BUDGET=False).")

In [ ]:
if RUN_DATA_BUDGET:
    res = pd.read_csv(RESULTS_PATH)
    pts = []
    for n_rows in BUDGET_PLAN:
        row = res[res.run_id == f"C2_n{n_rows}_evalmini"]
        if len(row):
            pts.append((n_rows, row.iloc[0]))
    full = res[res.run_id == "C2_evalmini"].iloc[0]
    pts.append((len(tr), full))
    xs = [p[0] for p in pts]
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    for a, col, title in zip(ax, ["bertscore_f1", "intent_macro_f1"], ["BERTScore-F1", "Intent macro-F1"]):
        a.plot(xs, [p[1][col] for p in pts], "o-", color="#2e7d32", label="fine-tuned (C2)")
        for cfg_, colr in (("B0", "#9aa5b1"), ("B1", "#6c8ebf")):
            ref = res[res.run_id == f"{cfg_}_evalmini"]
            if len(ref):
                a.axhline(ref.iloc[0][col], ls="--", color=colr, label=cfg_)
        a.set_xscale("log"); a.set_xlabel("training rows"); a.set_title(title + " (eval-mini)"); a.grid(alpha=.3); a.legend()
    plt.suptitle("Figure 3 (partial) - data budget"); plt.tight_layout()
    plt.savefig(D["figures"] / "fig3_data_budget_partial.png", dpi=170); plt.show()

## Done — what is in /kaggle/working now
`adapters/C2/final/` (the adapter, tokenizer files, `README.md` model card) · `adapters/C2/loss_log.csv` and `training_summary.json` · `config/qlora_config.json` · `preds/C2_*` · `results/results_log.csv` (B0, B1, C2 rows), `e1_partial_table.csv`, `systems_metrics.csv` · `figures/fig2_loss_curve.png`, `fig_c2_vs_baselines_test.png` · `docs/MODEL_CARD.md`.

**Keep your results:** use **Save Version → Save & Run All (Commit)** so the Output tab keeps `SahayakLLM/` and `C2_adapter.zip`. Download them (or attach this version's output to the next notebook) — the adapter is also on the Hugging Face Hub if you set `HF_ADAPTER_REPO`.\n\nNext: Phase B (agents, tools, policy index, scenario files in the repo) → the final evaluation / human-eval / demo notebook.